The **Intelligence**

In [1]:
from datetime import datetime, timedelta

# --- [CP-01] Parametric Observation Window ---

def get_observation_window():
    """FUNCTION 1: Get the 'Memory' parameter (weeks)"""
    # The Crux: We start with a 4-week baseline.
    # This is where the AI decides how far back to look.
    active_window_weeks = 4
    return active_window_weeks

def calculate_date_boundary(weeks):
    """FUNCTION 2: Calculate the 'Cut-off' date"""
    # The Crux: Today is March 1, 2026.
    # This function finds the date 28 days ago.
    today = datetime.now()
    start_date = today - timedelta(weeks=weeks)
    return start_date

# --- TEST THE LOGIC ---
window = get_observation_window()
boundary = calculate_date_boundary(window)

print(f"--- Your Partner in Kitchen: Memory Status ---")
print(f"Memory Window: {window} weeks")
print(f"Currently ignoring anything before: {boundary.strftime('%B %d, %Y')}")

--- Your Partner in Kitchen: Memory Status ---
Memory Window: 4 weeks
Currently ignoring anything before: February 01, 2026


In [2]:
# --- [CP-01] User Intelligence Profile Logic ---

class UserIntelligenceProfile:
    def __init__(self, user_id, active_window_size=4):
        self.user_id = user_id
        # AC: active_window_size (Integer) created
        self.active_window_size = active_window_size

    def get_window_start_date(self):
        """AC: System calculates $W_{obs}$ boundary for Sunday drafts"""
        today = datetime.now()
        # Logic: Current Date - (active_window_size * 7 days)
        w_obs_boundary = today - timedelta(weeks=self.active_window_size)
        return w_obs_boundary

# --- IMPLEMENTATION ---
# Imagine this is fetching from your backend config
profile = UserIntelligenceProfile(user_id="User_01", active_window_size=4)
w_obs = profile.get_window_start_date()

print(f"✅ AC 1: User_Intelligence_Profile initialized with size {profile.active_window_size}")
print(f"✅ AC 2: Sunday Drafts will ignore data older than: {w_obs.strftime('%Y-%m-%d')}")

✅ AC 1: User_Intelligence_Profile initialized with size 4
✅ AC 2: Sunday Drafts will ignore data older than: 2026-02-01


In [3]:
# --- BASELINE: [CP-01] Parametric Window ---

def run_sprint_1_check():
    window = get_observation_window()
    boundary = calculate_date_boundary(window)

    print(f"✅ User Story [CP-01] Closed.")
    print(f"Decision: The Partner will look back to {boundary.strftime('%Y-%m-%d')}.")
    print(f"Status: Logic is ready for Function 3 (Data Fetching).")

run_sprint_1_check()

✅ User Story [CP-01] Closed.
Decision: The Partner will look back to 2026-02-01.
Status: Logic is ready for Function 3 (Data Fetching).


logic for function 3 - rolling history

In [4]:
from datetime import datetime, timedelta
import json

# --- FN 2: CALCULATE BOUNDARY (The logic we baselined earlier) ---
def calculate_boundary(active_window_size=4):
    today = datetime.now()
    # Logic: Current Date - (W_obs * 7 days)
    return today - timedelta(weeks=active_window_size)

In [5]:
def fetch_rolling_history(user_id, start_date):
    vault_data = [
        {"meal_id": "M-101", "date": datetime(2026, 2, 25), "complexity_score": 8, "regional_tags": ["North Indian"]},
        {"meal_id": "M-102", "date": datetime(2026, 2, 20), "complexity_score": 4, "regional_tags": ["Continental"]},
        {"meal_id": "M-099", "date": datetime(2026, 1, 1), "complexity_score": 9, "regional_tags": ["Italian"]}
    ]

    # Filter and convert datetime to string so JSON can handle it
    active_history = []
    for item in vault_data:
        if item["date"] >= start_date:
            # We create a copy and convert the date to a string
            clean_item = item.copy()
            clean_item["date"] = item["date"].isoformat() # <--- THE FIX
            active_history.append(clean_item)

    return json.dumps(active_history)

# execution flow

In [6]:
# --- THE EXECUTION FLOW ---
# 1. Get the boundary first
w_obs_boundary = calculate_boundary(active_window_size=4)

# 2. Now pass it to the fetcher
rolling_json = fetch_rolling_history("Lead_Dev_01", w_obs_boundary)

print(f"✅ Boundary set to: {w_obs_boundary.strftime('%Y-%m-%d')}")
print(f"📦 Successfully fetched {len(json.loads(rolling_json))} records for analysis.")

✅ Boundary set to: 2026-02-01
📦 Successfully fetched 2 records for analysis.


In [7]:
# --- [CP-03] Function 4: Taste DNA Analysis ---

def analyze_taste_dna(json_data):
    """
    Function 3.1: Parses rolling history to find trends.
    Output: DNA Dictionary (Avg Complexity, Top Regions)
    """
    data = json.loads(json_data)

    if not data:
        return {"avg_complexity": 0, "top_regions": [], "status": "No data in window"}

    # 1. Calculate Average Complexity
    total_complexity = sum(item['complexity_score'] for item in data)
    avg_complexity = total_complexity / len(data)

    # 2. Extract and Flatten Regional Tags
    all_tags = []
    for item in data:
        all_tags.extend(item['regional_tags'])

    # 3. Find unique regions (Dominant Tags)
    unique_regions = list(set(all_tags))

    dna_profile = {
        "avg_complexity": round(avg_complexity, 2),
        "dominant_regions": unique_regions,
        "meal_count": len(data)
    }

    return dna_profile

# --- EXECUTION FLOW ---
# Pass the output from Function 3 into Function 4
user_dna = analyze_taste_dna(rolling_json)

print("--- 🧬 TASTE DNA PROFILE GENERATED ---")
print(f"Current Complexity Level: {user_dna['avg_complexity']}/10")
print(f"Active Regional Interests: {', '.join(user_dna['dominant_regions'])}")

--- 🧬 TASTE DNA PROFILE GENERATED ---
Current Complexity Level: 6.0/10
Active Regional Interests: Continental, North Indian


In [8]:
# --- [CP-03] Function 4: Behavior Drift Detector ---

def identify_behavior_drift(json_data):
    """
    Function 4: Detects significant shifts in cooking habits.
    If complexity changes by > 30%, it triggers a 'Drift'.
    """
    data = json.loads(json_data)
    if len(data) < 2: return False

    # 1. Split data into Current Week (Latest) and History (Previous)
    # (Assuming data is sorted by date)
    current_week = data[0:1] # Most recent meal
    previous_weeks = data[1:] # Older meals in window

    # 2. Compare Average Complexity
    curr_comp = sum(m['complexity_score'] for m in current_week) / len(current_week)
    prev_comp = sum(m['complexity_score'] for m in previous_weeks) / len(previous_weeks)

    # 3. Logic: If complexity shift is > 30%, habits have drifted
    drift_threshold = 0.30
    percent_change = abs(curr_comp - prev_comp) / prev_comp

    has_drifted = percent_change > drift_threshold

    if has_drifted:
        print("⚠️ DRIFT DETECTED: User habits are shifting significantly.")
    else:
        print("✅ STABLE: Habits are consistent with history.")

    return has_drifted

# --- EXECUTION ---
drift_signal = identify_behavior_drift(rolling_json)

⚠️ DRIFT DETECTED: User habits are shifting significantly.


In [9]:
# --- [CP-00] Function 5: Intelligence Profile Writer ---

def update_intelligence_profile(user_id, new_window_size, recalc_trigger):
    """
    Function 5: The 'Writer'. Saves parameters to the database.
    Input: user_id, new_window_size (the updated W_obs), recalc_trigger (timestamp)
    """

    # In production, this would be:
    # db.execute("UPDATE Profiles SET window = ?, last_recalc = ? WHERE id = ?", ...)

    intelligence_payload = {
        "user_id": user_id,
        "active_window_size": f"{new_window_size} weeks",
        "last_recalc_timestamp": recalc_trigger.strftime('%Y-%m-%d %H:%M:%S'),
        "status": "COMPLETED"
    }

    # Simulation of a successful DB write
    print(f"💾 DATABASE UPDATE SUCCESSFUL for {user_id}")
    print(f"📝 New Parameters: Window={new_window_size}, TriggeredAt={intelligence_payload['last_recalc_timestamp']}")

    return None # Output is void as per spec

# --- FINAL EXECUTION FLOW ---

# If drift was detected in Fn 4, we suggest a shorter window (e.g., 2 weeks)
suggested_window = 2 if drift_signal else 4

update_intelligence_profile(
    user_id="Vramanbalahm",
    new_window_size=suggested_window,
    recalc_trigger=datetime.now()
)

💾 DATABASE UPDATE SUCCESSFUL for Vramanbalahm
📝 New Parameters: Window=2, TriggeredAt=2026-03-01 12:05:00


GIT HUB commits

In [10]:
# --- [FINAL BASELINE] Pushing 5-Function Engine to GitHub ---

import os
import glob
from google.colab import userdata

# 1. AUTHENTICATION & REPO CONFIG
GITHUB_USER = "Vramanbalahm"
REPO_NAME = "project-momentum"
TOKEN = userdata.get('GITHUB_TOKEN')
REPO_URL = f"https://{TOKEN}@github.com/{GITHUB_USER}/{REPO_NAME}.git"

# 2. LOCATE THE FILE
notebooks = glob.glob("*.ipynb")
if not notebooks:
    print("❌ Error: No notebook file found. Please save/download your file first.")
else:
    target_file = notebooks[0]

    # 3. GIT EXECUTION
    !git config --global user.email "lead_dev@vraman.com"
    !git config --global user.name "Lead Developer"

    # Reset and Re-init for a clean baseline
    !rm -rf .git
    !git init
    !git remote add origin {REPO_URL}
    !git add "{target_file}"
    !git commit -m "feat: [CP-00] Full 5-Function Intelligence Engine Baseline"

    # Push to Main
    !git branch -M main
    !git push -u origin main --force

    print(f"\n🚀 SPRINT COMPLETE: {target_file} has been baselined to GitHub!")

❌ Error: No notebook file found. Please save/download your file first.
